# Phase 3: Fuzzy Systems — `FS_Model.ipynb`**Team 1 — NYC Taxi Trip Duration**

---## Words We Use in This Notebook (Glossary)Before reading the code, here are the main terms used in this phase.| Term | Meaning ||------|---------|| **Fuzzy Set** | A set where membership is not binary (0 or 1) but a degree between 0 and 1. For example, a 12 km trip might be 0.7 "medium" and 0.3 "long." || **Membership Function (MF)** | A curve that maps a crisp input value to a membership degree (0–1). Common shapes: triangular, trapezoidal, Gaussian. || **Linguistic Variable** | A variable described with words instead of numbers — e.g., trip distance is "short," "medium," or "long." || **Fuzzy Rule** | An IF–THEN statement using linguistic variables — e.g., IF distance IS long AND hour IS rush_hour THEN duration IS very_long. || **Fuzzy Inference System (FIS)** | The complete system: fuzzify inputs → apply rules → defuzzify to get a crisp predicted value. || **Defuzzification** | Converting fuzzy output back into a single number (e.g., centroid method). || **scikit-fuzzy** | Python library (`skfuzzy`) for building fuzzy inference systems. || **Interpretability** | How easy it is for a human to understand *why* the model made a prediction. Measured by rule count, rule length, and feature count. || **H3** | Phase 3 hypothesis: Does the fuzzy system provide meaningful interpretability while maintaining acceptable predictive performance compared to the NN (Phase 1) and EA (Phase 2)? |

---### What this notebook doesIn this notebook, we build a **Fuzzy Inference System (FIS)** to predict NYC taxi trip duration.Unlike the neural network (Phase 1) and the evolutionary algorithm (Phase 2), the fuzzy system is designed for **interpretability** — the goal is a model whose reasoning a human can read and understand, expressed as simple IF–THEN rules using everyday language like "short distance" or "rush hour."We intentionally trade some prediction accuracy for transparency. The fuzzy system will not beat the NN on raw metrics, and that is expected and acceptable. The value is in producing a model where every prediction can be explained.**Phases at a glance:**| Phase | Method | Notebook | Primary Goal ||-------|--------|----------|-------------|| 1 | Neural Network | `NN_Analysis.ipynb` | Maximize predictive accuracy || 2 | Evolutionary Algorithm | `EA_Optimization.ipynb` | Optimize NN hyperparameters via GA || 3 | **Fuzzy System** | **`FS_Model.ipynb` (this notebook)** | **Interpretable prediction via fuzzy rules** |

---## How to Run This Notebook1. **Open the notebook in NRP JupyterHub** (required for final results).   You may test locally, but all final outputs must be executed on NRP with the PyTorch2 environment.2. **Run cells in order from top to bottom.** Each cell depends on earlier cells.3. **Do not modify constants** (seed, split sizes, data path) — these must match Phases 1 and 2.### Execution environmentThe fuzzy system is implemented in **Python using scikit-fuzzy (`skfuzzy`)**, not TensorFlow or PyTorch. However, the notebook runs inside the NRP PyTorch2 environment as required by the project spec. The stack requirement is about the execution environment, not the modeling library.

---## Fair Evaluation RulesTo ensure honest and comparable results across all three phases, we follow strict data usage rules:- **Same dataset:** First 1,000,000 rows from `train.csv`- **Same split:** 70% train / 15% validation / 15% test (identical to Phases 1 and 2)- **Same seed:** All random operations use the same fixed seed as prior phases- **No re-shuffling:** Data order is identical to Phases 1 and 2- **Test set untouched:** The 15% test set is used **exactly once** for final evaluation in Step 3- **Validation only for design/tuning:** All membership function tuning and rule refinement uses the validation set- **No target leakage:** No feature depends on future or target information

---# Step 1: Fuzzy System Design & Setup## 1.1 — Fuzzy System Objective**Prediction target:** `trip_duration` (seconds), consistent with Phases 1 and 2.**Purpose:** Build a fuzzy inference system that provides **human-readable, rule-based predictions** for NYC taxi trip duration. The system intentionally trades some predictive accuracy for **interpretability** — every prediction can be traced back to a small set of IF–THEN rules using linguistically meaningful terms (e.g., "IF distance IS long AND pickup_hour IS rush_hour THEN duration IS very_long").**Hypothesis (H3):** The fuzzy system provides meaningful interpretability (measured by number of rules, average rule length, and feature coverage) while maintaining acceptable predictive performance compared to the Phase 1 NN and Phase 2 EA-optimized NN.**Evaluation path:**- **During development (Steps 1–2):** All design decisions, membership function tuning, and rule validation use the **validation set only**.- **Final evaluation (Step 3):** The fuzzy system is evaluated **once** on the held-out 15% test set, and metrics (RMSE, MAE, R², MAPE) are compared against Phase 1 and Phase 2 results.**What "acceptable" means:** We do not require the fuzzy system to match or exceed the NN's R². A meaningful drop in accuracy is justified if the model is fully transparent — i.e., a non-expert can read the rules and understand why a given trip was predicted to be long or short.

---## 1.2 — Feature Selection: Evidence from Phases 1 and 2### Why we select features carefullyFuzzy systems scale poorly with many inputs. Each input feature multiplies the potential rule space. The project FAQ explicitly warns about this. We target **6 input features** to keep the rule base manageable and interpretable.### Phase 1 NN Permutation Importance (Validation Set, R²_log drop)These are the features ranked by how much validation R² drops when the feature is randomly shuffled — higher drop means more important:| Rank | Feature | R²_log Drop | Std | Family ||------|---------|------------|-----|--------|| 1 | `haversine_km` | **2.0482** | 0.0064 | Spatial || 2 | `hour_cos` | 0.3014 | 0.0006 | Temporal || 3 | `delta_lat` | 0.2860 | 0.0011 | Spatial || 4 | `pickup_hour` | 0.1077 | 0.0007 | Temporal || 5 | `dow_cos` | 0.0846 | 0.0004 | Temporal || 6 | `delta_lon` | 0.0644 | 0.0010 | Spatial || 7 | `pickup_dow` | 0.0477 | 0.0003 | Temporal || 8 | `passenger_count` | 0.0038 | 0.0002 | Context || ... | `vendor_1/2`, `store_and_fwd`, `pickup_month` | < 0.004 | — | Various |### Grouped importance (NN)| Family | Total R²_log Drop | Interpretation ||--------|-------------------|---------------|| Spatial | **2.399** | Dominant — distance and direction are by far the strongest predictors || Temporal | **0.639** | Strong — time of day and day of week matter significantly || Context | **0.123** | Weak — vendor and passenger info contribute minimally |### Phase 2 EA findingsThe EA-optimized NN did **not** improve over the Phase 1 NN (ΔR² = −0.026, bootstrap CI includes 0). This confirms the Phase 1 feature set and architecture were already near-optimal, and reinforces that Phase 3 should focus on interpretability rather than accuracy gains.

---## 1.3 — Selected Input Features (6 features)Based on the Phase 1/2 evidence above, we select the following 6 features as fuzzy input variables:| # | Feature | Family | NN Importance Rank | R²_log Drop | Why Selected | Expected Influence on Duration ||---|---------|--------|-------------------|------------|-------------|-------------------------------|| 1 | `haversine_km` | Spatial | 1 | 2.048 | By far the most important feature across both models. Directly interpretable as "how far is the trip." | Longer distance → longer trip duration || 2 | `pickup_hour` | Temporal | 4 | 0.108 | Raw hour (0–23) maps naturally to fuzzy labels like "early morning," "morning commute," "midday," "evening rush," "night." More interpretable than sin/cos encoding. | Rush hours → longer trips due to traffic; late night → shorter trips || 3 | `pickup_dow` | Temporal | 7 | 0.048 | Day of week (0=Mon, 6=Sun) maps to "weekday" vs "weekend" patterns. More intuitive than sin/cos for fuzzy rules. | Weekdays (esp. Mon–Fri) → more traffic → longer trips || 4 | `delta_lat` | Spatial | 3 | 0.286 | North–south displacement. Captures directional travel patterns the NN found important (rank 3). | Large absolute displacement → longer trip || 5 | `delta_lon` | Spatial | 6 | 0.064 | East–west displacement. Together with delta_lat, captures travel direction without the complexity of haversine alone. | Large absolute displacement → longer trip || 6 | `passenger_count` | Context | 8 | 0.004 | Weakest selected feature, but the only interpretable context variable. Included to represent non-spatial, non-temporal factors. | Minimal direct effect; may correlate with trip type |### Features NOT selected (and why)| Feature | Why Excluded ||---------|-------------|| `hour_sin`, `hour_cos` | Cyclic encodings are mathematically useful for NNs but not linguistically interpretable. We use raw `pickup_hour` instead, which maps naturally to fuzzy labels. || `dow_sin`, `dow_cos` | Same reasoning — raw `pickup_dow` is more natural for fuzzy rules ("weekday" vs "weekend"). || `pickup_month` | Low importance in both models (NN rank 12, Ridge rank 5). Adds complexity without proportional interpretability gain. || `vendor_1`, `vendor_2` | Binary categorical. Not meaningful as fuzzy variables — vendor identity has no ordinal or continuous structure. || `store_and_fwd_Y` | Negligible importance (NN drop = 0.0001). Nearly zero variance. |

---## 1.4 — Output Variable Design**Output variable:** `predicted_duration` (seconds)The output uses **5 fuzzy labels** spanning typical NYC taxi trip durations. Boundaries are informed by the observed distribution in the training data (Phase 1), where most trips fall between 3–30 minutes with a right-skewed tail.| Label | Approximate Range (seconds) | Approximate Range (minutes) | Description ||-------|---------------------------|---------------------------|-------------|| `very_short` | 0 – 300 | 0 – 5 min | Quick trips: same-neighborhood, short hops || `short` | 120 – 900 | 2 – 15 min | Typical short urban trips || `medium` | 600 – 1800 | 10 – 30 min | Average-length trips, moderate distance || `long` | 1200 – 3600 | 20 – 60 min | Cross-borough or congested trips || `very_long` | 2400 – 5400+ | 40 – 90+ min | Airport runs, outer-borough, or heavy traffic |**Notes:**- Ranges deliberately **overlap** — this is fundamental to fuzzy logic. A 15-minute trip might be 0.4 "short" and 0.6 "medium" simultaneously.- Membership functions will be **triangular** by default (simplest, most interpretable). Brandon will refine shapes and exact boundaries from the actual training data distribution in Step 1.- **Defuzzification method:** Centroid (center of gravity) — the most common and interpretable approach.- The exact numeric boundaries will be finalized after Brandon completes the `membership_range_table` using observed data percentiles.

---
## 1.5 — Brandon Kirk Membership Range Table (Refined Step 1 Proposal)

This section converts the approved fuzzy variables into a **numerically disciplined design**. The goal is to keep the system interpretable, reproducible, and small enough to avoid rule explosion.

### Brandon design rules used
- Prefer **3 memberships per input** unless the feature clearly needs more granularity.
- Use **trapezoidal sets at the edges** when a boundary should stay "fully low" or "fully high" for a range of values.
- Use **triangular sets in the middle** when we want a simple transition region.
- Keep ranges broad enough to cover realistic NYC taxi behavior without overfitting to a few outliers.
- Treat these as the **initial locked Step 1 values**; Step 2 may make small adjustments after checking the actual training-set histograms, but not large conceptual changes.

### Refined membership range table

| Variable | Family | Universe / Numeric Range | Membership Labels | Membership Type | Proposed Parameters | Why this design is interpretable |
|---|---|---:|---|---|---|---|
| `haversine_km` | Spatial | 0 to 30 km | `short`, `medium`, `long` | trap / tri / trap | short = `[0, 0, 1.5, 4.0]`  •  medium = `[2.5, 7.5, 12.5]`  •  long = `[10.0, 16.0, 30.0, 30.0]` | Distance is the strongest predictor, so we keep 3 simple labels with overlap rather than many narrow buckets. |
| `pickup_hour` | Temporal | 0 to 23 hours | `early_morning`, `morning_rush`, `midday`, `evening_rush`, `night` | trap / tri / tri / tri / trap | early = `[0, 0, 5, 7]`  •  morning = `[6, 8, 10]`  •  midday = `[10, 13, 16]`  •  evening = `[15, 18, 20]`  •  night = `[19, 21, 23, 23]` | Hour needs 5 sets because traffic behavior changes more than a simple low/medium/high split would capture. |
| `pickup_dow` | Temporal | 0 to 6 | `weekday`, `weekend` | trap / trap | weekday = `[0, 0, 3.5, 4.5]`  •  weekend = `[4.5, 5.0, 6.0, 6.0]` | A binary weekday/weekend split is easy to explain and avoids overcomplicating a weak temporal effect. |
| `delta_lat` | Spatial | -0.20 to 0.20 degrees | `south`, `neutral`, `north` | tri / tri / tri | south = `[-0.20, -0.10, -0.01]`  •  neutral = `[-0.03, 0.00, 0.03]`  •  north = `[0.01, 0.10, 0.20]` | Signed latitude change naturally maps to south/neutral/north movement. |
| `delta_lon` | Spatial | -0.20 to 0.20 degrees | `west`, `neutral`, `east` | tri / tri / tri | west = `[-0.20, -0.10, -0.01]`  •  neutral = `[-0.03, 0.00, 0.03]`  •  east = `[0.01, 0.10, 0.20]` | Same structure as latitude keeps the design symmetric and easy to document. |
| `passenger_count` | Context | 1 to 6 passengers | `low`, `medium`, `high` | trap / tri / trap | low = `[1.0, 1.0, 1.5, 2.0]`  •  medium = `[1.5, 3.0, 4.5]`  •  high = `[4.0, 5.0, 6.0, 6.0]` | Passenger count is weak, so a compact 3-set design is enough. |
| `predicted_duration` | Output | 0 to 5400 sec | `very_short`, `short`, `medium`, `long`, `very_long` | trap / tri / tri / tri / trap | very_short = `[0, 0, 180, 420]`  •  short = `[240, 600, 960]`  •  medium = `[780, 1500, 2280]`  •  long = `[1800, 3000, 4200]`  •  very_long = `[3600, 4500, 5400, 5400]` | Five output labels provide interpretable verbal predictions without becoming too granular. |

### Boundary notes
- **Overlap is intentional.** A trip can be partly `short` and partly `medium` at the same time.
- **Distance cap:** 30 km is used as a practical fuzzy universe for most taxi trips; rare extreme trips can be clipped to the universe maximum.
- **Directional variables:** `delta_lat` and `delta_lon` are signed, so the fuzzy labels describe movement direction directly.
- **Passenger count:** values above 6 are not part of the normal taxi distribution used in this project and should be clipped if they appear.

### Why these shapes were chosen
- **Trapezoidal edge sets** help avoid awkward behavior at the minimum and maximum ends of a variable.
- **Triangular middle sets** keep the interpretation simple and make plots easy to explain in the report.
- **Symmetric directional sets** for latitude/longitude reduce unnecessary design complexity.

### Brandon deliverables completed here
- `membership_range_table` content is fully specified.
- Membership counts are locked.
- Initial boundaries are defined.
- Shape choices are justified.



---
## 1.6 — Brandon Interpretability Budget

Interpretability is the primary justification for Phase 3, so the fuzzy system must be constrained **before** implementation.

### Structural targets

| Metric | Locked Target | Why this limit exists |
|---|---:|---|
| **Number of input features** | 6 | Keeps the model within the team target and avoids a rule-space explosion. |
| **Memberships per input** | Mostly 3 | Three labels are usually enough to express low / medium / high style reasoning. |
| **Exceptions to 3 memberships** | `pickup_hour` may use 5; `pickup_dow` uses 2 | These are the only two variables where a different count clearly improves interpretability. |
| **Theoretical full rule space** | 810 | Computed as `3 × 5 × 2 × 3 × 3 × 3`; this is too large to use directly. |
| **Target active rule count** | 25 to 40 | Large enough for coverage, small enough to explain in a report. |
| **Hard ceiling on active rules** | 50 | Going above this would weaken interpretability. |
| **Target average rule length** | 2 to 3 antecedents | Shorter rules are easier to justify and easier to read. |
| **Hard ceiling on rule length** | 4 antecedents | Longer rules become hard to explain and maintain. |
| **Feature coverage goal** | Every input appears in at least 1 meaningful rule | Prevents selected features from becoming dead weight. |
| **Redundancy goal** | No near-duplicate rules | Avoids bloated rule bases that appear large without adding meaning. |

### Brandon rationale
The full combinatorial rule space is far too large to use directly. Therefore, the implementation should not generate every possible combination. Instead, Step 2 should build a **compact expert rule base** centered on the strongest and most interpretable interactions, especially:

- `haversine_km` + `pickup_hour`
- `haversine_km` + `pickup_dow`
- `haversine_km` + directional displacement
- selective use of `passenger_count` only where it adds meaning

### How this will be reported later
Step 3 should explicitly report:
- final number of active rules
- average rule length
- one or two plain-English example rules
- explanation of why the compact rule base supports interpretability better than the NN baseline



---## 1.7 — Methodology LockThe following constraints are locked for Phase 3 and must not be changed during implementation:| Constraint | Value | Reason ||-----------|-------|--------|| **Dataset** | First 1,000,000 rows of `train.csv` | Same as Phases 1 and 2 || **Split** | 70/15/15 train/val/test with same seed | Reuse exact Phase 1 split — no re-shuffling || **Test set** | Untouched until Step 3 final evaluation | Prevents overfitting to test data || **Validation use** | Design, tuning, rule refinement only | All Step 1–2 decisions validated here || **Implementation** | Python + scikit-fuzzy (`skfuzzy`) | NOT TensorFlow-only. The fuzzy system must be a real fuzzy inference system, not a neural surrogate. || **Execution environment** | NRP JupyterHub (PyTorch2 stack) | Stack requirement is about the environment, not the modeling library || **Random seed** | Same as Phases 1 and 2 | Reproducibility || **Defuzzification** | Centroid method | Most common, most interpretable |### Compliance statementThis Phase 3 implementation reuses the controlled data split from Phases 1 and 2 without re-shuffling. The test set remains untouched until the one-time final evaluation in Step 3. The fuzzy system is implemented as a genuine fuzzy inference system using scikit-fuzzy, running within the NRP PyTorch2 environment. No features introduce target leakage or depend on future information.

---
## Step 1 Complete — Handoff Summary

### Deliverables completed

| Deliverable | Status | Owner |
|---|---|---|
| FS objective statement | ✅ Complete (Section 1.1) | Nadir |
| Feature selection with evidence | ✅ Complete (Sections 1.2–1.3) | Nadir |
| Output variable design | ✅ Complete (Section 1.4) | Nadir |
| Refined membership range table | ✅ Complete (Section 1.5) | Brandon |
| Interpretability budget | ✅ Complete (Section 1.6) | Brandon |
| Methodology lock | ✅ Complete (Section 1.7) | Iran / David |

### What happens next (Step 2 — Implementation, due Apr 16)
- **Brandon:** Use the locked range table below to instantiate the scikit-fuzzy variables.
- **Dominic:** Convert these tables into polished final notebook/wiki presentation.
- **Iran:** Confirm no leakage and no split drift.
- **David:** Approve Step 1 signoff.
- **All:** Build the compact fuzzy rule base and validate on the untouched validation set.

### Brandon signoff note
Brandon's Step 1 work is complete once the notebook includes:
1. the numeric universe for every variable,
2. membership labels and shape types,
3. the interpretability budget,
4. a reproducible code cell that instantiates the fuzzy variables.

This notebook now contains all four.



------# Step 2: Fuzzy System Implementation*(Step 2 code cells begin below — due Thursday, April 16)*> **Status:** Awaiting Step 1 approval and Brandon's refined membership ranges before implementation begins.

---## Cell 1 — Imports, Constants & Environment> **What this will do:** Load all required libraries (numpy, pandas, skfuzzy, sklearn) and define constants matching Phases 1 and 2.

In [ ]:
# Step 2 — To be implemented after Step 1 approval# Imports and constants will go here# Must match Phase 1/2: SEED, NROWS, DATA_PATH, split ratios

---## Cell 2 — Data Load & Split Verification> **What this will do:** Load the same 1M rows, apply the same split, verify row counts match Phases 1 and 2.

In [ ]:
# Step 2 — To be implemented# Reuse exact data loading and splitting from Phase 1# Verify: train ~700k, val ~150k, test ~150k

---## Cell 3 — Feature Engineering (Same Pipeline as Phases 1 & 2)> **What this will do:** Apply the same `build_features()` function from Phase 1, then select only the 6 fuzzy input features.

In [ ]:
# Step 2 — To be implemented# Apply build_features() from Phase 1# Select only: haversine_km, pickup_hour, pickup_dow, delta_lat, delta_lon, passenger_count

---
## Cell 4 — Define Membership Functions (Ready to Run)
> **What this does:** Instantiates the six fuzzy inputs and one fuzzy output exactly as locked in Brandon's Step 1 design.



In [ ]:
# Step 2 — Brandon-ready membership function implementation
# Run this after imports and after setting up skfuzzy / ctrl imports in Cell 1.

import numpy as np
import pandas as pd
import skfuzzy as fuzz
import skfuzzy.control as ctrl

# -----------------------------
# 1) Universes
# -----------------------------
haversine_km = ctrl.Antecedent(np.arange(0, 30.01, 0.1), 'haversine_km')
pickup_hour = ctrl.Antecedent(np.arange(0, 24, 1), 'pickup_hour')
pickup_dow = ctrl.Antecedent(np.arange(0, 7, 1), 'pickup_dow')
delta_lat = ctrl.Antecedent(np.arange(-0.20, 0.201, 0.001), 'delta_lat')
delta_lon = ctrl.Antecedent(np.arange(-0.20, 0.201, 0.001), 'delta_lon')
passenger_count = ctrl.Antecedent(np.arange(1, 6.01, 0.1), 'passenger_count')
predicted_duration = ctrl.Consequent(np.arange(0, 5401, 1), 'predicted_duration')

# -----------------------------
# 2) Input membership functions
# -----------------------------
# haversine_km
haversine_km['short'] = fuzz.trapmf(haversine_km.universe, [0, 0, 1.5, 4.0])
haversine_km['medium'] = fuzz.trimf(haversine_km.universe, [2.5, 7.5, 12.5])
haversine_km['long'] = fuzz.trapmf(haversine_km.universe, [10.0, 16.0, 30.0, 30.0])

# pickup_hour
pickup_hour['early_morning'] = fuzz.trapmf(pickup_hour.universe, [0, 0, 5, 7])
pickup_hour['morning_rush'] = fuzz.trimf(pickup_hour.universe, [6, 8, 10])
pickup_hour['midday'] = fuzz.trimf(pickup_hour.universe, [10, 13, 16])
pickup_hour['evening_rush'] = fuzz.trimf(pickup_hour.universe, [15, 18, 20])
pickup_hour['night'] = fuzz.trapmf(pickup_hour.universe, [19, 21, 23, 23])

# pickup_dow
pickup_dow['weekday'] = fuzz.trapmf(pickup_dow.universe, [0, 0, 3.5, 4.5])
pickup_dow['weekend'] = fuzz.trapmf(pickup_dow.universe, [4.5, 5.0, 6.0, 6.0])

# delta_lat
delta_lat['south'] = fuzz.trimf(delta_lat.universe, [-0.20, -0.10, -0.01])
delta_lat['neutral'] = fuzz.trimf(delta_lat.universe, [-0.03, 0.00, 0.03])
delta_lat['north'] = fuzz.trimf(delta_lat.universe, [0.01, 0.10, 0.20])

# delta_lon
delta_lon['west'] = fuzz.trimf(delta_lon.universe, [-0.20, -0.10, -0.01])
delta_lon['neutral'] = fuzz.trimf(delta_lon.universe, [-0.03, 0.00, 0.03])
delta_lon['east'] = fuzz.trimf(delta_lon.universe, [0.01, 0.10, 0.20])

# passenger_count
passenger_count['low'] = fuzz.trapmf(passenger_count.universe, [1.0, 1.0, 1.5, 2.0])
passenger_count['medium'] = fuzz.trimf(passenger_count.universe, [1.5, 3.0, 4.5])
passenger_count['high'] = fuzz.trapmf(passenger_count.universe, [4.0, 5.0, 6.0, 6.0])

# -----------------------------
# 3) Output membership functions
# -----------------------------
predicted_duration['very_short'] = fuzz.trapmf(predicted_duration.universe, [0, 0, 180, 420])
predicted_duration['short'] = fuzz.trimf(predicted_duration.universe, [240, 600, 960])
predicted_duration['medium'] = fuzz.trimf(predicted_duration.universe, [780, 1500, 2280])
predicted_duration['long'] = fuzz.trimf(predicted_duration.universe, [1800, 3000, 4200])
predicted_duration['very_long'] = fuzz.trapmf(predicted_duration.universe, [3600, 4500, 5400, 5400])

predicted_duration.defuzzify_method = 'centroid'

# -----------------------------
# 4) Exportable table for documentation / QA
# -----------------------------
membership_range_table = pd.DataFrame([
    ['haversine_km', 'Spatial', '0 to 30 km', 'short, medium, long', 'trap / tri / trap', '[0,0,1.5,4.0] | [2.5,7.5,12.5] | [10,16,30,30]'],
    ['pickup_hour', 'Temporal', '0 to 23', 'early_morning, morning_rush, midday, evening_rush, night', 'trap / tri / tri / tri / trap', '[0,0,5,7] | [6,8,10] | [10,13,16] | [15,18,20] | [19,21,23,23]'],
    ['pickup_dow', 'Temporal', '0 to 6', 'weekday, weekend', 'trap / trap', '[0,0,3.5,4.5] | [4.5,5,6,6]'],
    ['delta_lat', 'Spatial', '-0.20 to 0.20', 'south, neutral, north', 'tri / tri / tri', '[-0.20,-0.10,-0.01] | [-0.03,0.00,0.03] | [0.01,0.10,0.20]'],
    ['delta_lon', 'Spatial', '-0.20 to 0.20', 'west, neutral, east', 'tri / tri / tri', '[-0.20,-0.10,-0.01] | [-0.03,0.00,0.03] | [0.01,0.10,0.20]'],
    ['passenger_count', 'Context', '1 to 6', 'low, medium, high', 'trap / tri / trap', '[1,1,1.5,2] | [1.5,3,4.5] | [4,5,6,6]'],
    ['predicted_duration', 'Output', '0 to 5400 sec', 'very_short, short, medium, long, very_long', 'trap / tri / tri / tri / trap', '[0,0,180,420] | [240,600,960] | [780,1500,2280] | [1800,3000,4200] | [3600,4500,5400,5400]'],
], columns=['variable', 'family', 'range', 'labels', 'membership_type', 'parameters'])

membership_range_table



---## Cell 5 — Define Fuzzy Rule Base> **What this will do:** Create the IF–THEN fuzzy rules based on domain knowledge and data patterns.

In [ ]:
# Step 2 — To be implemented# Define fuzzy rules using ctrl.Rule()# Target: 30-50 meaningful rules with 2-3 antecedents each

---## Cell 6 — Build & Run Fuzzy Inference System> **What this will do:** Create the control system, run predictions on validation set, compute metrics.

In [ ]:
# Step 2 — To be implemented# Build ControlSystem and ControlSystemSimulation# Predict on validation set# Compute RMSE, MAE, R², MAPE on validation